Задача

Необходимо создать комплекс моделей, которые помогут заказчику в разработке сервиса для определения стереотипа личности на основе фото.


Детализация

1. Валидация изображения.

Проверить:
1. файл представлен в графическим формате (не допускаются архивы типа RAR, ZIP и т. п.); 
2. на изображении есть лицо;
3. на изображении нет закрытых областей лица (очки, маски, головные уборы);
4. лицо ориентировано на камеру (анфас, допускается небольшое отклонение, субъективно – не более 15°);
5. выражение лица нейтрально, отсутствуют яркие эмоции (опционально);
6. изображение не сгенерировано и не подвергалось обработке (опционально);
7. кадрировать изображение (оставить голову, можно с небольшим запасом вокруг);
8. размер кадрированного изображения не меньше минимально допустимого;
   

Порядок проверки/обработки можно менять.
Определить наиболее оптимальный порядок.
При необходимости можно объединять пункты.
Можно добавить новые пункты, обосновав их полезность.

2. Извлечение данных из изображения.

При успешной валидации из изображения необходимо извлечь:
пол;
возраст;
группа (выполнить кластеризацию для создания групп) (опционально);
метаданные (EXIF, геолокация и т. п.) (опционально).

Можно предложить другие признаки, которые можно извлечь из изображения.


Результат

Пайплайн: изображения на входе, таблица извлеченных признаков на выходе.

__Извлеченные признаки необходимо накапливать в таблице: одна строка – одно изображение.__ 
Для изображений, не прошедших валидацию в столбце “rejected” необходимо указать причину.

Заказчик может оценить результат на тестовых данных.

Данные

Предоставляется стартовый датасет.
Также предоставляется список рекомендуемых открытых датасетов, которые участники могут использовать при желании.
К данным можно добавлять собственную разметку.
Можно расширять набор данных, используя любые открытые источники.

Рекомендуемый стек

FastAI
YOLO
CLIP
DeepFace / RetinaFace
UMAP
Label Studio / Roboflow Annotate / CVAT

Приобретаемые навыки

В результате работы над проектом, студенты смогут: 

Развить навык классификации изображений:
- используя знакомые способы (PyTorch, TensorFlow);
- с FastAI;
- с YOLO;
- zero-shot с CLIP или подобной.
Познакомиться с детекцией объектов:
- разметка данных;
- YOLO (подготовка данных, обучение, валидация, визуалиация);
- кадрирование (подготовка данных для следующего этапа).
- Разобраться с разметкой изображений для задачи object detection и потренироваться с ручной разметкой или автоматизированной.
- Развить навык исследовательского анализа: например, сравнить качество классификации на исходных изображениях и на кадрированных, сравнить качество модели на данных с ручной и автоматизированной разметкой, сравнить затрачиваемые ресурсы.
- Приветствуются выдвижение и проверка различных гипотез, любые исследования.
- Разработать подход, который облегчит проведение экспериментов (смена моделей, датасетов, предобработки изображений и т. п., выдача результата в удобной для последующей обработки форме).
- Можно обратить внимание на ClearML или подобные инструменты.


In [1]:
pip install ultralytics 

Note: you may need to restart the kernel to use updated packages.


In [2]:
import tempfile
from pathlib import Path

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import supervision as sv
import random
from ultralytics import YOLO
from PIL import Image, ImageColor

In [3]:
CWD = Path.cwd()
input_images = CWD.joinpath('src')  # путь изображению
output_images = CWD.joinpath('frames') # путь для сохранения результата

In [4]:
import cv2
import os
import sys
from ultralytics import YOLO
import numpy as np 

MODEL_PATH = "yolov11l-face.pt" 
MARGIN = 10
FACE_CLASS_ID = 0 # ID класса для лица человека в yolov-face.pt это 0
                  
CONFIDENCE_THRESHOLD = 0.6 # Минимальная уверенность для детекции

def process_images_in_folder(input_folder, output_folder, model, padding_percent=0.3, frame_color=(0, 255, 0), frame_thickness=2):
    """
    Обрабатывает все .jpg/.jpeg изображения в папке с использованием YOLO:
    находит одно лицо (или объект класса FACE_CLASS_ID), кадрирует его с отступом,
    рисует рамку на кадрированном изображении и сохраняет в выходную папку.

    Args:
        input_folder (str): Путь к папке с исходными изображениями.
        output_folder (str): Путь к папке для сохранения результатов.
        model (YOLO): Загруженная модель YOLO.
        padding_percent (float): Процент от ширины/высоты лица для отступа.
        frame_color (tuple): Цвет рамки в формате BGR.
        frame_thickness (int): Толщина рамки.
    """
    # проверка и создание выходной папки
    if not os.path.isdir(input_folder):
        print(f"Ошибка: Входная папка не найдена или не является папкой: {input_folder}")
        return
    os.makedirs(output_folder, exist_ok=True)

    print(f"Начинаю обработку изображений из папки: {input_folder}")
    print(f"Использую модель: {MODEL_PATH}, Класс объекта: {FACE_CLASS_ID}, Порог уверенности: {CONFIDENCE_THRESHOLD}")
    print(f"Результаты будут сохранены в: {output_folder}")

    processed_count = 0
    skipped_no_face = 0
    skipped_multi_face = 0
    skipped_error = 0
    frame_count = 0

    processed_data = {}
    # итерация по файлам во входной папке
    for filename in os.listdir(input_folder):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):            
            input_image_path = input_folder.joinpath(filename)
            
            output_filename = f"cropped_{filename}"
            # cохраняем в том же формате, что и прочитали 
            output_image_path = output_folder.joinpath(output_filename)
            
            print(f"Обработка файла: {filename}")

            # загрузка изображения
            image = cv2.imread(input_image_path)
            if image is None:
                print(f"  Ошибка: Не удалось загрузить изображение {filename}")
                skipped_error += 1
                continue

            # обнаружение объектов с помощью YOLO
            try:
                results = model.predict(source=image, 
                                        conf=CONFIDENCE_THRESHOLD, 
                                        classes=[FACE_CLASS_ID], 
                                        verbose=False) # verbose=False убирает лишний вывод YOLO
            except Exception as e:
                print(f"  Ошибка при выполнении model.predict для {filename}: {e}")
                skipped_error += 1
                continue
            
            boxes = results[0].boxes.xyxy.cpu().numpy() # координаты [x1, y1, x2, y2]
            
            num_faces = len(boxes)

            # проверка количества найденных лиц/объектов
            if num_faces == 0:
                print(f"  Объект класса {FACE_CLASS_ID} не найден на изображении: {filename}")
                skipped_no_face += 1
                continue
            elif num_faces > 1:
                print(f"  Найдено несколько объектов ({num_faces}). Пропускаем: {filename}")
                skipped_multi_face += 1
                continue

            # кадрирование (с отступом)
            # берем координаты единственного найденного объекта
            x1, y1, x2, y2 = boxes[0].astype(int)
            x, y = x1, y1
            w = x2 - x1
            h = y2 - y1

            if w <= 0 or h <= 0:
                 print(f"  Ошибка: Некорректные размеры рамки (w={w}, h={h}) для {filename}")
                 skipped_error += 1
                 continue

            pad_x = int(w * padding_percent) # отступ сторон
            pad_y = int(h * padding_percent) # отступ сверху/снизу

            # координаты кадрирования в исходном изображении
            # c отступом 
            crop_x1 = (x1 - pad_x)
            crop_y1 = (y1 - pad_x)
            crop_x2 = (x2 + pad_y)
            crop_y2 = (y2 + pad_y)
          

            # обрезка кадра
            cropped_image = image[crop_y1:crop_y2, crop_x1:crop_x2]  
            
            if cropped_image.size == 0:
                print(f"  Ошибка: Не удалось вырезать область (возможно, объект у края): {filename}")
                skipped_error += 1
                continue

            # рисование рамки на кадрированном изображении
            # координаты лица относительно кадрированного изображения
            face_x_in_crop = x - crop_x1
            face_y_in_crop = y - crop_y1
            
            # рисуем прямоугольник (используем w и h исходного бокса)
            cv2.rectangle(cropped_image,
                          (face_x_in_crop, face_y_in_crop),
                          (face_x_in_crop + w, face_y_in_crop + h),
                          frame_color,
                          frame_thickness)

            processed_data[output_filename] = (crop_x1, crop_y1, crop_x2, crop_y2)    
            
            # сохранение результата
            try:
                success = cv2.imwrite(output_image_path, cropped_image)
                if success:
                    print(f"  Успешно сохранено: {output_filename}")
                    processed_count += 1
                else:
                    # ветка  срабатывает, если imwrite вернул False
                    print(f"  Ошибка: cv2.imwrite не удалось сохранить файл {output_filename}")
                    skipped_error += 1
            except Exception as e:
                print(f"  Исключение при сохранении файла {output_filename}: {e}")
                skipped_error += 1
        else:
             if os.path.isfile(input_folder.joinpath(filename)):
                print(f"\nПропускаем файл с неподдерживаемым расширением: {filename}")

    # вывод статистики
    print("--- Обработка завершена ---")
    print(f"Всего обработано файлов (с 1 объектом): {processed_count}")
    print(f"Пропущено (объект не найден): {skipped_no_face}")
    print(f"Пропущено (несколько объектов): {skipped_multi_face}")
    print(f"Пропущено (ошибки загрузки/обработки/сохранения): {skipped_error}")
    print(f"Результаты сохранены в папке: {output_folder}")

    return processed_data

In [5]:
if __name__ == "__main__":
    INPUT_DIR = CWD.parent.joinpath('filter_image') # папка JPG/JPEG/PNG изображениями
    
    # создадим тестовые папки, если их нет
    if not os.path.exists(INPUT_DIR):
        os.makedirs(INPUT_DIR)
        print(f"Создана папка '{INPUT_DIR}'. Поместите в неё ваши изображения.")
    if not os.path.exists(output_images):
        os.makedirs(output_images)
    
    try:
        print(f"Загрузка модели YOLO из {MODEL_PATH}...")
        yolo_model = YOLO(MODEL_PATH)
        print("Модель YOLO успешно загружена.")
    except Exception as e:
        print(f"Критическая ошибка: Не удалось загрузить модель YOLO из {MODEL_PATH}: {e}")
        sys.exit(1) # выход если модель не загрузилась

    # запуск функции поиска лица на изображении
    boxes = process_images_in_folder(
        INPUT_DIR,
        output_images,
        yolo_model,
        padding_percent=0.1, 
        frame_thickness=3)
    

Загрузка модели YOLO из yolov11l-face.pt...
Модель YOLO успешно загружена.
Начинаю обработку изображений из папки: /home/anton/projects/scanFace/filter_image
Использую модель: yolov11l-face.pt, Класс объекта: 0, Порог уверенности: 0.6
Результаты будут сохранены в: /home/anton/projects/scanFace/detect_face/frames
Обработка файла: face-2615.jpg
  Успешно сохранено: cropped_face-2615.jpg
Обработка файла: 1a786966-5139-4c5a-8ada-346b62eb4bd1.jpg
  Успешно сохранено: cropped_1a786966-5139-4c5a-8ada-346b62eb4bd1.jpg
Обработка файла: face-1440.jpg
  Успешно сохранено: cropped_face-1440.jpg
Обработка файла: face-924.jpg
  Успешно сохранено: cropped_face-924.jpg
Обработка файла: face-4976.jpg
  Успешно сохранено: cropped_face-4976.jpg
Обработка файла: face-47.jpg
  Успешно сохранено: cropped_face-47.jpg
Обработка файла: face-1736.jpg
  Успешно сохранено: cropped_face-1736.jpg
Обработка файла: face-3808.jpg
  Успешно сохранено: cropped_face-3808.jpg
Обработка файла: face-22.jpg
  Успешно сохране

In [6]:
# Text drawing parameters
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.7 
TEXT_COLOR = (0, 255, 255) 
TEXT_THICKNESS = 2
TEXT_POSITION = (10, 30)

In [7]:
# подключаем библиотеку компьютерного зрения 
import cv2

# точно так же загружаем модели для определения пола и возраста
genderProto=CWD.joinpath("gender_deploy.prototxt")
genderModel=CWD.joinpath("gender_net.caffemodel")
ageProto=CWD.joinpath("age_deploy.prototxt")
ageModel=CWD.joinpath("age_net.caffemodel")

# настраиваем свет
MODEL_MEAN_VALUES=(78.4263377603, 87.7689143744, 114.895847746)
# итоговые результаты работы нейросетей для пола и возраста
genderList=['Male ','Female']
ageList=['(0-2)', '(4-6)', '(8-12)', '(15-20)', '(25-32)', '(38-43)', '(48-53)', '(60-100)']

# запускаем нейросети по определению пола и возраста
genderNet=cv2.dnn.readNet(genderModel,genderProto)
ageNet=cv2.dnn.readNet(ageModel,ageProto)

image_directory = CWD.joinpath('frames')
image_files = list(image_directory.glob('*.[jp][pn]g'))

results = []
    
for img_name, box in boxes.items():
    face = cv2.imread(output_images.joinpath(img_name))
    # получаем на этой основе новый бинарный пиксельный объект
    blob=cv2.dnn.blobFromImage(face, 1.0, (227,227), MODEL_MEAN_VALUES, swapRB=False)
    # отправляем его в нейросеть для определения пола
    genderNet.setInput(blob)
    # получаем результат работы нейросети
    genderPreds=genderNet.forward()
    # выбираем пол на основе этого результата
    gender=genderList[genderPreds[0].argmax()]
    # отправляем результат в переменную с полом
    print(f'Gender: {gender}')

    # делаем то же самое для возраста
    ageNet.setInput(blob)
    agePreds=ageNet.forward()
    age=ageList[agePreds[0].argmax()]
    print(f'Age: {age[1:-1]} years')

    # # добавляем текст возле каждой рамки в кадре
    # cv2.putText(face, f'{gender}, {age}', (box[0], box[1]), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,255), 2, cv2.LINE_AA)
    # # выводим итоговую картинку
    # cv2.imshow("Detecting age and gender", face)
    # добавляем текст возле каждой рамки в кадре
    cv2.putText(face, f'{gender}, {age}', TEXT_POSITION, FONT, FONT_SCALE, TEXT_COLOR, TEXT_THICKNESS, cv2.LINE_AA)
    success = cv2.imwrite(output_images.joinpath(img_name),face)
    print('Изображение сохранено', output_images.joinpath(img_name))           

Gender: Female
Age: 25-32 years
Изображение сохранено /home/anton/projects/scanFace/detect_face/frames/cropped_face-2615.jpg
Gender: Male 
Age: 38-43 years
Изображение сохранено /home/anton/projects/scanFace/detect_face/frames/cropped_1a786966-5139-4c5a-8ada-346b62eb4bd1.jpg
Gender: Male 
Age: 25-32 years
Изображение сохранено /home/anton/projects/scanFace/detect_face/frames/cropped_face-1440.jpg
Gender: Male 
Age: 48-53 years
Изображение сохранено /home/anton/projects/scanFace/detect_face/frames/cropped_face-924.jpg
Gender: Male 
Age: 38-43 years
Изображение сохранено /home/anton/projects/scanFace/detect_face/frames/cropped_face-4976.jpg
Gender: Male 
Age: 25-32 years
Изображение сохранено /home/anton/projects/scanFace/detect_face/frames/cropped_face-47.jpg
Gender: Female
Age: 25-32 years
Изображение сохранено /home/anton/projects/scanFace/detect_face/frames/cropped_face-1736.jpg
Gender: Female
Age: 8-12 years
Изображение сохранено /home/anton/projects/scanFace/detect_face/frames/crop